In [1]:
!pip install "numpy<2"

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
import pickle
import warnings
warnings.filterwarnings('ignore')

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

In [4]:
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split as sklearn_train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error


In [5]:
!pip install scikit-surprise

In [6]:
from surprise import Dataset, Reader, SVD
from surprise.prediction_algorithms.matrix_factorization import SVD
from surprise.model_selection import train_test_split, cross_validate
from surprise.accuracy import mse, mae, rmse
from surprise import accuracy

In [8]:
movies_data = pd.read_csv('/kaggle/input/datasets/omarabdelhamedomar/movie-lens/movies.csv')
rating_df = pd.read_csv('/kaggle/input/datasets/omarabdelhamedomar/movie-lens/ratings.csv')

In [9]:
print("Rating dataset shape:", rating_df.shape)
print("Movies dataset shape:", movies_data.shape)

Rating dataset shape: (100836, 4)
Movies dataset shape: (9742, 3)


In [10]:
print(f"\nRating range: {rating_df['rating'].min()} - {rating_df['rating'].max()}")
print(f"Number of unique users: {rating_df['userId'].nunique()}")
print(f"Number of unique movies: {rating_df['movieId'].nunique()}")
print(f"Total ratings: {len(rating_df)}")


Rating range: 0.5 - 5.0
Number of unique users: 610
Number of unique movies: 9724
Total ratings: 100836


In [11]:
all_genres = []
for genres in movies_data['genres'].dropna():
    if genres != '(no genres listed)':
        all_genres.extend(genres.split('|'))

genre_counts = pd.Series(all_genres).value_counts()
print(f"\nTotal unique genres: {len(genre_counts)}")
print("\nTop 10 genres:")
print(genre_counts.head(10))


Total unique genres: 19

Top 10 genres:
Drama        4361
Comedy       3756
Thriller     1894
Action       1828
Romance      1596
Adventure    1263
Crime        1199
Sci-Fi        980
Horror        978
Fantasy       779
Name: count, dtype: int64


In [12]:
total_possible_ratings = rating_df['userId'].nunique() * rating_df['movieId'].nunique()
actual_ratings = len(rating_df)
sparsity = (1 - actual_ratings / total_possible_ratings) * 100
print(f"\nData sparsity: {sparsity:.2f}%")


Data sparsity: 98.30%


In [13]:
print(f"\nAverage ratings per user: {rating_df.groupby('userId').size().mean():.2f}")
print(f"Average ratings per movie: {rating_df.groupby('movieId').size().mean():.2f}")


Average ratings per user: 165.30
Average ratings per movie: 10.37


In [14]:
most_rated = rating_df.groupby('movieId').size().sort_values(ascending=False).head(10)
most_rated_with_titles = most_rated.reset_index()
most_rated_with_titles = most_rated_with_titles.merge(movies_data[['movieId', 'title']], on='movieId')
print("\nMost rated movies:")
print(most_rated_with_titles)


Most rated movies:
   movieId    0                                      title
0      356  329                        Forrest Gump (1994)
1      318  317           Shawshank Redemption, The (1994)
2      296  307                        Pulp Fiction (1994)
3      593  279           Silence of the Lambs, The (1991)
4     2571  278                         Matrix, The (1999)
5      260  251  Star Wars: Episode IV - A New Hope (1977)
6      480  238                       Jurassic Park (1993)
7      110  237                          Braveheart (1995)
8      589  224          Terminator 2: Judgment Day (1991)
9      527  220                    Schindler's List (1993)


In [15]:
avg_ratings = rating_df.groupby('movieId').agg({'rating': ['mean', 'count']}).round(2)
avg_ratings.columns = ['avg_rating', 'num_ratings']
avg_ratings = avg_ratings.reset_index()
popular_movies = avg_ratings[avg_ratings['num_ratings'] >= 50].sort_values('avg_rating', ascending=False).head(10)
popular_movies = popular_movies.merge(movies_data[['movieId', 'title']], on='movieId')
print("\nHighest rated movies (min 50 ratings):")
print(popular_movies)


Highest rated movies (min 50 ratings):
   movieId  avg_rating  num_ratings  \
0      318        4.43          317   
1      858        4.29          192   
2      750        4.27           97   
3     1276        4.27           57   
4     2959        4.27          218   
5     1221        4.26          129   
6      904        4.26           84   
7    48516        4.25          107   
8     1213        4.25          126   
9    58559        4.24          149   

                                               title  
0                   Shawshank Redemption, The (1994)  
1                              Godfather, The (1972)  
2  Dr. Strangelove or: How I Learned to Stop Worr...  
3                              Cool Hand Luke (1967)  
4                                  Fight Club (1999)  
5                     Godfather: Part II, The (1974)  
6                                 Rear Window (1954)  
7                               Departed, The (2006)  
8                                  

In [16]:
rating_df['timestamp'] = rating_df['timestamp'].astype('int64')

print("Data quality check:")
print(f"Duplicate ratings: {rating_df.duplicated().sum()}")
print(f"Users with only 1 rating: {sum(rating_df.groupby('userId').size() == 1)}")
print(f"Movies with only 1 rating: {sum(rating_df.groupby('movieId').size() == 1)}")

Data quality check:
Duplicate ratings: 0
Users with only 1 rating: 0
Movies with only 1 rating: 3446


In [17]:
min_user_ratings = 20
min_movie_ratings = 20

user_counts = rating_df['userId'].value_counts()
movie_counts = rating_df['movieId'].value_counts()

valid_users = user_counts[user_counts >= min_user_ratings].index
valid_movies = movie_counts[movie_counts >= min_movie_ratings].index

filtered_ratings = rating_df[
    (rating_df['userId'].isin(valid_users)) & 
    (rating_df['movieId'].isin(valid_movies))
]

print(f"\nAfter filtering:")
print(f"Original ratings: {len(rating_df)}")
print(f"Filtered ratings: {len(filtered_ratings)}")
print(f"Users: {rating_df['userId'].nunique()} -> {filtered_ratings['userId'].nunique()}")
print(f"Movies: {rating_df['movieId'].nunique()} -> {filtered_ratings['movieId'].nunique()}")


After filtering:
Original ratings: 100836
Filtered ratings: 67898
Users: 610 -> 610
Movies: 9724 -> 1297


In [18]:
rating_df_clean = filtered_ratings.copy()

In [19]:
user_encoder = LabelEncoder()
movie_encoder = LabelEncoder()

In [20]:
rating_df_clean['user_encoded'] = user_encoder.fit_transform(rating_df_clean['userId'])
rating_df_clean['movie_encoded'] = movie_encoder.fit_transform(rating_df_clean['movieId'])

n_users = rating_df_clean['user_encoded'].nunique()
n_movies = rating_df_clean['movie_encoded'].nunique()

In [21]:
print(f"\nEncoded dimensions:")
print(f"Number of users: {n_users}")
print(f"Number of movies: {n_movies}")


Encoded dimensions:
Number of users: 610
Number of movies: 1297


In [22]:
train_data, test_data = sklearn_train_test_split(rating_df_clean, test_size=0.2, random_state=42)

print(f"\nTrain set size: {len(train_data)}")
print(f"Test set size: {len(test_data)}")


Train set size: 54318
Test set size: 13580


In [23]:
train_data, test_data = sklearn_train_test_split(rating_df_clean, test_size=0.2, random_state=42)

print(f"\nTrain set size: {len(train_data)}")
print(f"Test set size: {len(test_data)}")


Train set size: 54318
Test set size: 13580


In [25]:
reader = Reader(rating_scale=(0.5, 5.0))
surprise_data = Dataset.load_from_df(rating_df_clean[['userId', 'movieId', 'rating']], reader)

In [26]:
trainset, testset = train_test_split(surprise_data, test_size=0.2, random_state=29)

In [27]:
svd_model = SVD(n_factors=100, n_epochs=20, biased=True, random_state=42)
svd_model.fit(trainset)

In [28]:
svd_model = SVD(n_factors=100, n_epochs=20, biased=True, random_state=42)
svd_model.fit(trainset)

In [30]:
svd_predictions = svd_model.test(testset)

In [31]:
svd_rmse = rmse(svd_predictions, verbose=False)
svd_mae = mae(svd_predictions, verbose=False)

print(f"SVD RMSE: {svd_rmse:.4f}")
print(f"SVD MAE: {svd_mae:.4f}")

SVD RMSE: 0.8383
SVD MAE: 0.6421


In [32]:
cv_results = cross_validate(svd_model, surprise_data, measures=['RMSE', 'MAE'], cv=5, verbose=False)
print(f"Cross-validation RMSE: {cv_results['test_rmse'].mean():.4f}")
print(f"Cross-validation MAE: {cv_results['test_mae'].mean():.4f}")

Cross-validation RMSE: 0.8455
Cross-validation MAE: 0.6473


In [33]:
with open('svd_model.pkl', 'wb') as f:
    pickle.dump(svd_model, f)
print("SVD model saved as 'svd_model.pkl'")

SVD model saved as 'svd_model.pkl'
